In [1]:
import pandas as pd
import numpy as np

# Visualization (already used for EDA, useful again for evaluation)
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
# Imputation
from sklearn.impute import SimpleImputer
# Multicollinearity check
from statsmodels.stats.outliers_influence import variance_inflation_factor
# Modeling
from sklearn.linear_model import LogisticRegression
# Evaluation (for later stages — threshold tuning, calibration)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve
)
from sklearn.calibration import calibration_curve
# Feature scaling (logistic regression benefits from this, especially with regularization)
from sklearn.preprocessing import StandardScaler

In [2]:
df=pd.read_csv("Diabetes_new.csv")
df.head()

,Unnamed: 0,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,SkinThickness_missing,Insulin_missing
0,0,6,148,72,35,0,33.6,0.627,50,1,0,1
1,1,1,85,66,29,0,26.6,0.351,31,0,0,1
2,2,8,183,64,0,0,23.3,0.672,32,1,1,1
3,3,1,89,66,23,94,28.1,0.167,21,0,0,0
4,4,0,137,40,35,168,43.1,2.288,33,1,0,0


In [3]:
df = df.drop('Unnamed: 0', axis=1)
df.sample(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,SkinThickness_missing,Insulin_missing
53,8,176,90,34,300,33.7,0.467,58,1,0,0
335,0,165,76,43,255,47.9,0.259,26,0,0,0
691,13,158,114,0,0,42.3,0.257,44,1,1,1
423,2,115,64,22,0,30.8,0.421,21,0,0,1
605,1,124,60,32,0,35.8,0.514,21,0,0,1
663,9,145,80,46,130,37.9,0.637,40,1,0,0
384,1,125,70,24,110,24.3,0.221,25,0,0,0
188,8,109,76,39,114,27.9,0.640,31,1,0,0
761,9,170,74,31,0,44.0,0.403,43,1,0,1
342,1,0,68,35,0,32.0,0.389,22,0,0,1


# Feature selection

In [4]:
X=df.drop("Outcome", axis=1)
y=df["Outcome"]

In [5]:
print(X.columns.tolist())
print(X.shape, y.shape)

['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'SkinThickness_missing', 'Insulin_missing']
(768, 10) (768,)


# Test Train and Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(614, 10) (154, 10)
Outcome
0    0.651466
1    0.348534
Name: proportion, dtype: float64
Outcome
0    0.649351
1    0.350649
Name: proportion, dtype: float64


### Both sets are near-identical to the original 65.1/34.9 split — stratification worked exactly as intended.

## Now we can handle those missing values before fitting model.For that we are going to apply a uniform(Median) method at first though missingness is of multiple magnitudes.

In [8]:
cols_to_fix = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
X_train[cols_to_fix]=X_train[cols_to_fix].replace(0, np.nan)
X_test[cols_to_fix]=X_test[cols_to_fix].replace(0, np.nan)

In [9]:
# Impute using train-only statistics
imputer= SimpleImputer(strategy='median')
X_train[cols_to_fix]=imputer.fit_transform(X_train[cols_to_fix])
X_test[cols_to_fix]=imputer.fit_transform(X_test[cols_to_fix])

In [10]:
print(X_train[cols_to_fix].isnull().sum())
print(X_test[cols_to_fix].isnull().sum())

Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
dtype: int64
Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
dtype: int64


In [11]:
print(X_train[cols_to_fix].isnull().sum())
print(X_test[cols_to_fix].isnull().sum())

Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
dtype: int64
Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
dtype: int64


### One practical thing to build into your deployment pipeline:
    You need to save the imputer object (e.g. with pickle or joblib) after fitting it on training data, so at inference time you're reusing the exact same fitted medians — not refitting on whatever new data comes in

In [12]:
import joblib
joblib.dump(imputer, 'imputer.pkl')

['imputer.pkl']

# Scaling

### Technically StandardScaler will scale them along with everything else if you pass the whole X_train in as above (it'll transform 0/1 into something like -0.6/1.4 based on their mean/std). This isn't wrong, but it does slightly change how to think about the flag: it stops being a clean binary indicator and becomes just another standardized number. Most practitioners are fine with this for logistic regression — it won't break anything or bias results meaningfully — but if you want to keep the flags interpretable as clean binary features, you can scale only the continuous columns and leave the binary flags untouched:

In [13]:
scaler = StandardScaler()
continuous_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
flag_cols = ['SkinThickness_missing', 'Insulin_missing']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

In [14]:
print(X_train_scaled[continuous_cols].mean().round(2))
print(X_train_scaled[continuous_cols].std().round(2))

Pregnancies                -0.0
Glucose                    -0.0
BloodPressure               0.0
SkinThickness              -0.0
Insulin                    -0.0
BMI                         0.0
DiabetesPedigreeFunction   -0.0
Age                        -0.0
dtype: float64
Pregnancies                 1.0
Glucose                     1.0
BloodPressure               1.0
SkinThickness               1.0
Insulin                     1.0
BMI                         1.0
DiabetesPedigreeFunction    1.0
Age                         1.0
dtype: float64


In [15]:
# Save the model
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

# Fit Model

In [16]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

### That's the baseline fit — no regularization tuning yet, just default settings, so you have something working end-to-end before refining it.

In [17]:
coef_df = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)
print(coef_df)

                    feature  coefficient
1                   Glucose     1.196361
5                       BMI     0.695890
9           Insulin_missing     0.382144
0               Pregnancies     0.370842
6  DiabetesPedigreeFunction     0.250345
7                       Age     0.127859
3             SkinThickness     0.031471
4                   Insulin    -0.035753
2             BloodPressure    -0.063758
8     SkinThickness_missing    -0.137771


### This gives you the raw coefficients — we'll convert these to odds ratios in the next step, but first let's see if it runs cleanly and what the coefficient signs/magnitudes look like. Which features come out with the largest positive coefficients (increasing diabetes odds), and which are negative?

# Converting to odds ratios

### First, an important nuance: since you scaled the continuous features, these coefficients aren't directly interpretable as "odds ratios per raw unit" anymore — they represent the change per 1 standard deviation increase in that feature. That's actually useful here because it lets you compare relative importance across features fairly (Glucose and Pregnancies are now on the same footing), which raw-unit coefficients wouldn't give you.

In [18]:
coef_df['odds_ratio'] = np.exp(coef_df['coefficient'])
print(coef_df.sort_values('odds_ratio', ascending=False))

                    feature  coefficient  odds_ratio
1                   Glucose     1.196361    3.308056
5                       BMI     0.695890    2.005493
9           Insulin_missing     0.382144    1.465423
0               Pregnancies     0.370842    1.448955
6  DiabetesPedigreeFunction     0.250345    1.284468
7                       Age     0.127859    1.136393
3             SkinThickness     0.031471    1.031971
4                   Insulin    -0.035753    0.964879
2             BloodPressure    -0.063758    0.938232
8     SkinThickness_missing    -0.137771    0.871298


### For Glucose (coefficient 1.196), the odds ratio would be exp(1.196) ≈ 3.31 — meaning a 1 standard deviation increase in Glucose is associated with roughly a 3.3x increase in the odds of diabetes, holding other features constant. That's a strong, clinically sensible signal — glucose is the direct diagnostic marker for diabetes, so this is reassuring rather than surprising.

## What stands out in our results

1. **Glucose dominates** (**1.20**) — expected and clinically correct.

2. **BMI is the second strongest** (**0.70**) — also expected, as obesity is a well-known diabetes risk factor.

3. **`Insulin_missing`** (**0.38**) outperforms **`Insulin`** itself (**-0.036**) — this is the most interesting finding.

   Remember your earlier EDA, where missing **Insulin** values were associated with a slightly higher diabetes rate? This coefficient confirms that the pattern remains **even after controlling for all other features** in a multivariable model—not just in a simple two-variable comparison.

   This provides stronger evidence for your **Missing Not At Random (MNAR)** hypothesis than the raw percentage difference observed during EDA, making it an important finding to highlight in your report.

4. **`SkinThickness_missing`** has a **small negative coefficient** (**-0.14**). Its effect is much weaker than the **Insulin** missingness indicator, so it is worth mentioning but not overemphasizing.

5. **`Insulin`** (raw value) has **almost no predictive effect** (**-0.036**), meaning its contribution is essentially negligible.

   This is understandable because nearly half of the missing values were imputed using the same median value, reducing the variability in the feature. As a result, the **missingness indicator** carries more predictive information than the **imputed insulin values** themselves.